In [0]:
spark.sql("USE CATALOG e_comm")
spark.sql("USE SCHEMA bronze")

In [0]:
from pyspark.sql.functions import *
import pandas as pd
import requests
from io import StringIO
from delta.tables import DeltaTable

In [0]:
spark.sql("create table if not exists e_comm.bronze.customers(customer_id string, customer_unique_id string, customer_zip_code_prefix bigint, customer_city string, customer_state string, merge_flag boolean, ingestion_ts timestamp) using delta ")

In [0]:
from pyspark.sql.functions import lit, current_timestamp
import requests
import pandas as pd
from io import StringIO
url = "https://raw.githubusercontent.com/deepakmali17/E_commerce_dataplatform/refs/heads/main/datasets/customers.csv"

response = requests.get(url)

response.raise_for_status()
pdf  = pd.read_csv(StringIO(response.text))
df = spark.createDataFrame(pdf)
df_source = df.withColumn("merge_flag", lit(False)).withColumn("ingestion_ts", current_timestamp())



df_target = DeltaTable.forName(spark, "e_comm.bronze.customers")

df_target.alias("t").merge(df_source.alias("s"), "t.customer_id = s.customer_id")\
    .whenMatchedUpdateAll(
        condition = "t.customer_unique_id <> s.customer_unique_id or t.customer_zip_code_prefix <> s.customer_zip_code_prefix or t.customer_city <> s.customer_city or t.customer_state <> s.customer_state")\
    .whenNotMatchedInsertAll().execute()



In [0]:
%sql
select count(*) from e_comm.bronze.customers